In [1]:
import umap
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from sklearn.cluster import KMeans
import requests
import gzip
import pickle
from io import BytesIO
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from zadu import zadu

from pcc import PCC, PCUMAP

def download_and_load_dataset(url):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with gzip.open(BytesIO(response.content), "rb") as f:
        data = pickle.load(f)
    return data


def plot_embeddings(embeddings_2d, X, y, method, downsample_for_metrics=16):
    spec = [{
        "id"    : "tnc"
    },
    {"id": "mrre"},
    {"id": "pr"},
    {"id": "srho"}]
    X = np.float32(X)
    y = list(map(int, y))
    # Visualize the result
    plt.figure(figsize=(10, 8))
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=y, cmap='Spectral', s=5)
    plt.colorbar(boundaries=np.arange(11)-0.5).set_ticks(np.arange(10))

    local_keys = ['trustworthiness', 'continuity', 'mrre_false', 'mrre_missing']
    global_keys = ['pr', 'srho']

    all_scores = zadu.ZADU(spec, X[::downsample_for_metrics, :]).measure(embeddings_2d[::downsample_for_metrics, :])
    scores = []
    local_scores = []
    global_scores = []
    for s in all_scores:
        for k, v in s.items():
            if k in local_keys:
                local_scores.append(f"{k}: {v:.3f}")
            else:
                global_scores.append(f"{k}: {v:.3f}")

    scores_str = 'Global metrics: ' + ' '.join(global_scores) + '\n' + 'Local metrics: ' + ' '.join(local_scores)

    plt.title(f'{method}\n{scores_str}')
    plt.xlabel(f'{method} 1')
    plt.ylabel(f'{method} 2')

# --- Download Macosko data ---
url_macosko = "http://file.biolab.si/opentsne/benchmark/macosko_2015.pkl.gz"
data_macosko = download_and_load_dataset(url_macosko)

x_macosko = data_macosko["pca_50"].astype("float32")
y_macosko = data_macosko["CellType1"].astype(str)
y_macosko_encoded = LabelEncoder().fit_transform(y_macosko)
X, y = x_macosko, y_macosko_encoded

/home/jacob/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
np.random.seed(0)

clusters = []
n_clusters_list = [4, 8]
for n_clusters in n_clusters_list:
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init="auto")
    cluster_labels = kmeans.fit_predict(X)
    clusters.append(cluster_labels)

In [3]:
pcc_reducer = PCC(n_components=10, num_epochs=20, num_points=10, pearson=True, spearman=False, beta=5, k_epoch=2)
pcc_embedding = pcc_reducer.fit_transform(X, clusters)

100%|██████████| 20/20 [00:00<00:00, 64.70it/s]


In [4]:
pcc_embedding.shape

(44808, 10)

In [5]:
from pcc import PCC, PCUMAP
pcumap_reducer = PCUMAP(device='cuda', n_components=5)

In [6]:
pcumap_embedding = pcumap_reducer.fit_transform(X)

got embeddings
[TorchDR] Out of memory encountered, setting backend to 'keops' for UMAPAffinityIn object.


In [7]:
pcumap_embedding.shape

(44808, 5)